<a href="https://colab.research.google.com/github/Zahra-Mhdi/Deep-Learning-Exercises/blob/main/Session_8_Seq2Seq_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell 1 — Imports & Device :

In [53]:
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


Cell 2 — Dataset (Build) :

In [54]:
random.seed(42)
torch.manual_seed(42)

rooms = ["kitchen", "bedroom", "living room", "bathroom"]
temps = list(range(16, 31))

on_templates = [
    "turn on the {room} light",
    "switch on the {room} light",
    "turn the {room} light on",
]

off_templates = [
    "turn off the {room} light",
    "switch off the {room} light",
    "turn the {room} light off",
]

temp_templates = [
    "set temperature to {v} degrees",
    "set temp to {v}",
]

def norm_room(r):
    return r.upper().replace(" ", "_")

pairs = []

for r in rooms:
    for t in on_templates:
        pairs.append((t.format(room=r), f"INTENT=LIGHT_ON ROOM={norm_room(r)}"))
    for t in off_templates:
        pairs.append((t.format(room=r), f"INTENT=LIGHT_OFF ROOM={norm_room(r)}"))

for v in temps:
    for t in temp_templates:
        pairs.append((t.format(v=v), f"INTENT=SET_TEMP VALUE={v}"))

random.shuffle(pairs)
split = int(0.85 * len(pairs))
train_pairs = pairs[:split]
test_pairs = pairs[split:]


Cell 3 — Vocabulary & Tokenization :

In [55]:
def tok(s):
    return s.lower().split()

SPECIAL = ["<pad>", "<sos>", "<eos>", "<unk>"]

inp_vocab = SPECIAL + sorted({w for x,_ in train_pairs for w in tok(x)})
out_vocab = SPECIAL + sorted({w for _,y in train_pairs for w in y.split()})

inp_stoi = {w:i for i,w in enumerate(inp_vocab)}
out_stoi = {w:i for i,w in enumerate(out_vocab)}
out_itos = {i:w for w,i in out_stoi.items()}


Cell 4 — Dataset & DataLoader :

In [56]:
class CmdDS(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        x,y = self.pairs[i]
        xi = [inp_stoi["<sos>"]] + [inp_stoi.get(w,3) for w in tok(x)] + [inp_stoi["<eos>"]]
        yi = [out_stoi["<sos>"]] + [out_stoi.get(w,3) for w in y.split()] + [out_stoi["<eos>"]]
        return torch.tensor(xi, dtype=torch.long), torch.tensor(yi, dtype=torch.long)

def collate(b):
    xs,ys = zip(*b)
    xl = max(len(x) for x in xs)
    yl = max(len(y) for y in ys)
    xp = torch.full((len(xs), xl), inp_stoi["<pad>"], dtype=torch.long)
    yp = torch.full((len(ys), yl), out_stoi["<pad>"], dtype=torch.long)
    for i,(x,y) in enumerate(zip(xs,ys)):
        xp[i,:len(x)] = x
        yp[i,:len(y)] = y
    return xp, yp

train_loader = DataLoader(CmdDS(train_pairs), batch_size=32, shuffle=True, collate_fn=collate)
test_loader  = DataLoader(CmdDS(test_pairs), batch_size=32, shuffle=False, collate_fn=collate)


 Cell 5 — Seq2Seq Model (Embedding + Encoder + Decoder) :

In [57]:
class Encoder(nn.Module):
    def __init__(self, v, e, h):
        super().__init__()
        self.emb = nn.Embedding(v, e, padding_idx=inp_stoi["<pad>"])
        self.rnn = nn.GRU(e, h, batch_first=True)
    def forward(self, x):
        _,h = self.rnn(self.emb(x))
        return h

class Decoder(nn.Module):
    def __init__(self, v, e, h):
        super().__init__()
        self.emb = nn.Embedding(v, e, padding_idx=out_stoi["<pad>"])
        self.rnn = nn.GRU(e, h, batch_first=True)
        self.fc = nn.Linear(h, v)
    def forward(self, y, h):
        o,_ = self.rnn(self.emb(y), h)
        return self.fc(o)

class Seq2Seq(nn.Module):
    def __init__(self, enc, dec):
        super().__init__()
        self.enc = enc
        self.dec = dec
    def forward(self, x, y):
        h = self.enc(x)
        return self.dec(y[:,:-1], h)


Cell 6 — Build Model & Optimizer :

In [58]:
model = Seq2Seq(
    Encoder(len(inp_vocab), 64, 128),
    Decoder(len(out_vocab), 64, 128)
)

opt = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss(ignore_index=out_stoi["<pad>"])


Cell 7 — Train :

In [59]:
for _ in range(80):
    for x,y in train_loader:
        opt.zero_grad()
        out = model(x,y)
        loss = loss_fn(out.reshape(-1,out.size(-1)), y[:,1:].reshape(-1))
        loss.backward()
        opt.step()


Cell 8 — Predict (Greedy) :

In [60]:
def predict(s):
    x = torch.tensor([[inp_stoi["<sos>"]] + [inp_stoi.get(w, inp_stoi["<unk>"]) for w in tok(s)] + [inp_stoi["<eos>"]]], dtype=torch.long)
    h = model.enc(x)
    y = torch.tensor([[out_stoi["<sos>"]]], dtype=torch.long)
    res = []
    for _ in range(10):
        o = model.dec(y, h)
        t = o[:,-1].argmax(-1).item()
        w = out_itos[t]
        if w == "<eos>":
            break
        if w not in ("<sos>", "<pad>"):
            res.append(w)
        y = torch.cat([y, torch.tensor([[t]], dtype=torch.long)], 1)
    return " ".join(res)


Cell 9 — Test (Accuracy) :

In [61]:
correct = 0
total = 0

for x,y in test_loader:
    for i in range(x.size(0)):
        inp_words = [inp_vocab[t] for t in x[i].tolist() if t not in (inp_stoi["<pad>"],)]
        s = " ".join([w for w in inp_words if w not in ("<sos>", "<eos>", "<pad>")])
        pred = predict(s)
        gold_words = [out_vocab[t] for t in y[i].tolist() if t not in (out_stoi["<pad>"],)]
        gold = " ".join([w for w in gold_words if w not in ("<sos>", "<eos>", "<pad>")])
        correct += int(pred.strip() == gold.strip())
        total += 1

print(correct / total)


0.4444444444444444


Cell 10 — Final Input/Output :

In [62]:
lines = []
while True:
    s = input()
    if s.strip() == "":
        break
    lines.append(s)

for l in lines:
    print(predict(l))



switch on the bathroom light
turn the living room light off
set temp to 19

INTENT=LIGHT_ON ROOM=BATHROOM
INTENT=LIGHT_OFF ROOM=LIVING_ROOM
INTENT=SET_TEMP VALUE=19
